# RQ2: pure random retention — 288 and 64 input tokens

Use **Internet and T4 ×2**, the existing **rq1-ih-inputs-v1** private Dataset,
and the locked original ApET environment. Push the new code first.

Six questions / five images; baseline576, native ApET with/without merging at288/64,
and pure random at both budgets with three fixed seeds. **Pure random uses no
FPS, reconstruction-error calculation, basis replacement or merging.** All decoder
pruning is off. It uses the native full projection and a selection mask, so this
calibration does not measure compression overhead or a paper's average-layer budget.

Default False prevents accidental background execution after import. Set
`RUN_RANDOM_CALIBRATION = True` in the first code cell, then run the cells in order.
Same attached input as recent RQ2; no new ZIP upload. Calibration checks execution,
not a required accuracy. The broader TextVQA100 runner is a later step.
See `research/RQ2_PURE_RANDOM_CALIBRATION.md`.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_RANDOM_CALIBRATION = False  # Set True: six questions; both 288 and 64 input tokens.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None  # Existing rq1-ih-inputs-v1 Dataset/ZIP; not the older rq1-inputs-v1.
CHECKPOINT_ARGS = []  # Optional existing original-format model and vision paths.

if not RUN_RANDOM_CALIBRATION:
    raise RuntimeError("Set RUN_RANDOM_CALIBRATION = True, then run cells manually.")
print("Preparing pure-random calibration code; model inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True, timeout=120)
if not (repo / "scripts/run_rq2_random_calibration.py").is_file():
    raise RuntimeError("Push the new pure-random files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
rq2_random_runs = []
print("Repository:", repo, flush=True)


## Prepare the existing input Dataset and locked runtime

This reuses the original environment/cache in a live session. Independent notebook sessions do not share virtual environments or `/tmp` checkpoints.


In [ ]:
sys.path.insert(0, str(repo / "scripts"))
from rq1_input_ih import locate_inputs
inputs = locate_inputs(
    [Path("/kaggle/input"), Path("/kaggle/working")],
    Path("/kaggle/working/rq1-ih-inputs-v1"), explicit=INPUT_ZIP,
)
print("Accepted input directory:", inputs, flush=True)
setup_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
setup_log = Path(f"/kaggle/working/rq2-random-setup-{setup_stamp}.log")
runtime_report = Path(f"/kaggle/working/rq2-random-runtime-{setup_stamp}.json")

def stream_command(command, stream, *, cwd=None):
    print("Running:", " ".join(map(str, command)), flush=True)
    stream.write("COMMAND " + repr(command) + "\n")
    stream.flush()
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
    lines = []
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
            lines.append(line)
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    stream.write(f"EXIT {status}\n")
    stream.flush()
    if status:
        raise RuntimeError(f"Command failed (exit {status}); preserve {stream.name} before changing anything.")
    return "".join(lines)

with setup_log.open("x", encoding="utf-8") as stream:
    stream_command([
        "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
        "--output", str(runtime_report),
    ], stream, cwd=repo)
print("Locked original runtime ready. Run the prerequisite/helper cell next.", flush=True)
print("Setup log:", setup_log)
print("Runtime report:", runtime_report)


## Prerequisites and streaming helper

Tiny tests exercise the actual native code, pure-random FPS/solve guards, native64 preparation/generation, fixed scoring and restoration. Zero skips are required.


In [ ]:
def verify_rq2_random_prerequisites(*, refresh_runtime=False):
    """Require actual original-runtime tests, with no missing-Torch skips."""
    globals()["rq2_random_prerequisites_passed"] = False
    with setup_log.open("a", encoding="utf-8") as stream:
        if refresh_runtime:
            stream_command([str(original_python), "-X", "utf8", "-I", "scripts/check_original_env.py",
                "--require-cuda", "--expected-gpus", "2", "--output", str(runtime_report)], stream, cwd=repo)
        stream_command([str(original_python), "-X", "utf8", "-I", "-c",
                        "import torch; print(torch.__version__)"], stream, cwd=repo)
        output = stream_command([
            str(original_python), "-X", "utf8", "-m", "unittest",
            "research.tests.test_rq2_native", "research.tests.test_rq2_random_protocol",
            "research.tests.test_rq2_random_native", "research.tests.test_rq2_random_runner", "-v",
        ], stream, cwd=repo)
        count = re.search(r"Ran (\d+) tests?", output)
        if not count or int(count.group(1)) == 0 or re.search(r"skipped\s*=", output):
            raise RuntimeError("Prerequisites require actual tests with zero skips; keep the setup log.")
    globals()["rq2_random_prerequisites_passed"] = True
    print("Pure-random/native64 prerequisites passed.", flush=True)


def run_rq2_random_calibration():
    if not globals().get("rq2_random_prerequisites_passed", False):
        raise RuntimeError("Run verify_rq2_random_prerequisites() successfully before loading pretrained weights.")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path(f"/kaggle/working/rq2-pure-random-calibration-v1-{stamp}")
    archive, log = folder.with_suffix(".zip"), folder.with_suffix(".log")
    globals().setdefault("rq2_random_runs", []).append(folder)
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_rq2_random_calibration.py",
        "--inputs", str(inputs), "--output", str(folder), "--calibration",
        "--require-cuda", "--expected-gpus", "2", "--download-budget-gib", "20",
        "--setup-log", str(setup_log), "--runtime-report", str(runtime_report), *CHECKPOINT_ARGS]
    print("Six questions, eleven primary conditions each; parity/repeats are separate controls.", flush=True)
    print("Run directory:", folder, flush=True)
    fallback_archive = False
    try:
        with log.open("x", encoding="utf-8") as stream:
            stream_command(command, stream, cwd=repo)
    finally:
        if not archive.exists():
            fallback_archive = True
            fallback = folder / "notebook-fallback"
            fallback.mkdir(parents=True, exist_ok=True)
            (fallback / "README.txt").write_text(
                "Incomplete notebook fallback: runner did not create its normal ZIP.\n"
                "This does not establish successful calibration.\n", encoding="utf-8")
            for path in [setup_log, runtime_report, log]:
                if path.is_file():
                    shutil.copy2(path, fallback / path.name)
            with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as bundle:
                for path in sorted(folder.rglob("*")):
                    if path.is_file():
                        bundle.write(path, path.relative_to(folder))
        print("Download:", archive, "and", log, flush=True)
        if (folder / "report.json").is_file():
            report = json.loads((folder / "report.json").read_text())
            for key in ["status", "calibration_passed", "wall_seconds", "progress", "summaries", "error"]:
                if key in report:
                    print(key + ":", report[key], flush=True)
    if fallback_archive:
        raise RuntimeError("Runner ZIP missing; preserve the incomplete fallback ZIP/log.")
    report = json.loads((folder / "report.json").read_text())
    if report.get("version") != "rq2-pure-random-calibration-v1" or report.get("status") != "passed" or not report.get("calibration_passed"):
        raise RuntimeError("Calibration did not pass; download the ZIP/log for review.")
    print("Calibration complete. Review ZIP/log before the whole TextVQA100 experiment.", flush=True)
    return folder

# Re-executing this helper cell invalidates any previous prerequisite pass.
globals()["rq2_random_prerequisites_passed"] = False


In [ ]:
verify_rq2_random_prerequisites()


## Run the six-question calibration

One model load. 66 primary predictions, plus full576 parity and declared fixed/fresh controls. Keep all results regardless of which method wins.


In [ ]:
random_calibration_dir = run_rq2_random_calibration()


## Download before stopping the session

Download the timestamped **ZIP and matching log**. Setup/runtime evidence is included in the ZIP; links are also provided separately. If a prerequisite fails before this cell, download its setup log. Do not start a broad run until calibration review.


In [ ]:
from IPython.display import FileLink, display
paths = [setup_log, runtime_report]
for folder in rq2_random_runs:
    paths.extend([folder.with_suffix(".zip"), folder.with_suffix(".log")])
for path in paths:
    if path.exists():
        display(FileLink(str(path)))
